# Maillard Formulation Screening Example

This notebook demonstrates how to programmatically use the Maillard framework to screen a candidate formulation without using the CLI.

In [ ]:
import sys
from pathlib import Path
import os

project_root = str(Path(os.getcwd()).parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.conditions import ReactionConditions
from src.pipeline import MaillardPipeline
from src.usability_reports import DomainOfValidityChecker, build_confidence_package


## 1. Define Reaction Conditions


In [ ]:
conditions = ReactionConditions(
    pH=6.5, 
    temperature_celsius=140.0,
    water_activity=0.95,
    protein_type="pea_iso"
)


## 2. Define Formulation & Designer


In [ ]:
formulation = {
    "name": "Pea Isolate Test",
    "sugars": ["ribose", "glucose"],
    "amino_acids": ["cysteine", "leucine"],
    "lipids": ["hexanal"],
    "additives": [],
    "molar_ratios": {"ribose": 0.5, "glucose": 0.2, "cysteine": 0.2, "leucine": 0.1},
    "ph": 6.5,
    "temp": 140.0,
    "aw": 0.95,
    "time_minutes": 45.0,
    "protein_type": "pea_iso",
    "denaturation_state": 0.6,
    "catalyst": None
}

designer = MaillardPipeline(target_tag="meaty", minimize_tag="beany")


## 3. Run Prediction


In [ ]:
print("Running Generative Maillard Pipeline...")
result = designer.evaluate_single(formulation, conditions)

print(f"Target Score (Meaty):  {result.target_score:.2f}")
print(f"Risk Penalty (Beany):  {result.off_flavour_risk:.2f}")
print(f"Lipid Trapping Effic:  {result.trapping_efficiency:.1f}%")


## 4. Inspect Predicted Compounds


In [ ]:
print("Predicted Target Compounds:")
for target in result.targets:
    label = target['target'].label if target['target'] else "Unknown"
    print(f"- {label} ({target['type']}): {target.get('sensory', 'No sensory data')}")


## 5. Domain of Validity Check


In [ ]:
checker = DomainOfValidityChecker(target_tag="meaty")
precursors = formulation["sugars"] + formulation["amino_acids"] + formulation["lipids"]
warnings = checker.check(
    precursor_names=precursors,
    protein_type=formulation["protein_type"],
    temp_c=formulation["temp"],
    ph=formulation["ph"]
)

result.confidence_metadata = build_confidence_package(
    result, warnings, precursor_names=precursors,
    protein_type=formulation["protein_type"],
    formulation=formulation,
    baseline_conditions=conditions,
    designer=designer
)

print(f"Confidence Tier: {result.confidence_metadata['tier']}")
if warnings:
    print("\nWarnings:")
    for w in warnings:
        print(f"- {w.description}")
